In [ ]:
import pandas as pd

X_train = pd.read_csv("../data/X_train_tree.csv")

X_test = pd.read_csv(
    "../data/X_test_tree.csv"
).squeeze()

In [5]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [6]:
from xgboost import XGBRegressor

from src.evaluate import rmse_cv

In [7]:
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb_rmse = rmse_cv(
    xgb_model,
    X_train,
    y_train
).mean()

print(
    f"XGBoost RMSE: {xgb_rmse:.5f}"
)

XGBoost RMSE: 0.11822


In [8]:
X_test = pd.read_csv(
    "../data/X_test.csv"
)

test = pd.read_csv(
    "../data/test.csv"
)

print(X_test.shape)
print(test.shape)

(1459, 318)
(1459, 80)


In [9]:
xgb_model.fit(
    X_train,
    y_train
)

print("XGB 訓練完成")

XGB 訓練完成


In [13]:
submission.to_csv(
    "../submissions/xgb_submission.csv",
    index=False
)

print("XGB Submission 已建立")

XGB Submission 已建立


In [14]:
# XGBoost v2

xgb_model_v2 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v2 = rmse_cv(
    xgb_model_v2,
    X_train,
    y_train
).mean()

print(
    f"XGB v2 RMSE: {xgb_rmse_v2:.5f}"
)

XGB v2 RMSE: 0.11332


In [15]:
xgb_model_v2.fit(
    X_train,
    y_train
)

print("XGB v2 訓練完成")

XGB v2 訓練完成


In [19]:
submission_v2.to_csv(
    "../submissions/xgb_v2_submission.csv",
    index=False
)

print("XGB v2 Submission 已建立")

XGB v2 Submission 已建立


In [20]:
# XGBoost v3

xgb_model_v3 = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v3 = rmse_cv(
    xgb_model_v3,
    X_train,
    y_train
).mean()

print(
    f"XGB v3 RMSE: {xgb_rmse_v3:.5f}"
)

XGB v3 RMSE: 0.11448


In [21]:
xgb_model_v4 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)
xgb_rmse_v4 = rmse_cv(
    xgb_model_v4,
    X_train,
    y_train
).mean()

print(
    f"XGB v4 RMSE: {xgb_rmse_v4:.5f}"
)

XGB v4 RMSE: 0.11275


In [26]:
import numpy as np
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_grid = {
    "n_estimators": np.arange(1500, 3500, 200),
    "learning_rate": [0.01, 0.015, 0.02, 0.025],
    "max_depth": [3, 4],
    "subsample": [0.75, 0.8, 0.85],
    "colsample_bytree": [0.75, 0.8, 0.85],
    "min_child_weight": [1, 2, 3],
}

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,   # 先用10
    scoring="neg_root_mean_squared_error",
    cv=kf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("開始搜索...")

random_search.fit(
    X_train,
    y_train
)

print("\n=== 搜索完成 ===")
print(random_search.best_params_)
print(
    f"RMSE: {-random_search.best_score_:.5f}"
)

開始搜索...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

=== 搜索完成 ===
{'subsample': 0.75, 'n_estimators': np.int64(2300), 'min_child_weight': 2, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.85}
RMSE: 0.11366


In [27]:
xgb_best = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)

xgb_best.fit(
    X_train,
    y_train
)

print("最佳 XGB 訓練完成")

最佳 XGB 訓練完成


In [28]:
xgb_best_pred_log = xgb_best.predict(
    X_test
)

xgb_best_pred = np.expm1(
    xgb_best_pred_log
)

In [29]:
submission_best = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_best_pred
})

submission_best.head()

,Id,SalePrice
0,1461,48749.410156
1,1462,52409.851562
2,1463,47687.640625
3,1464,48707.074219
4,1465,48894.914062


In [30]:
submission_best.to_csv(
    "../submissions/xgb_random_search_submission.csv",
    index=False
)

print("XGB Random Search Submission 已建立")

XGB Random Search Submission 已建立


In [31]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

model = XGBRegressor(
    n_estimators=3500,
    learning_rate=0.015,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    gamma=0.01,
    reg_alpha=0.01,
    reg_lambda=1.5,
    random_state=42
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse = np.sqrt(
    -cross_val_score(
        model,
        X_train,
        y_train,
        scoring="neg_mean_squared_error",
        cv=kf
    )
)

print(rmse.mean())

0.11378364130756217


In [1]:
print(X_train.shape)
print(X_test.shape)

NameError: name 'X_train' is not defined